In [20]:
import os
import shutil
import subprocess
import numpy as np
import datetime

def edit_line(filepath, line_idx, new_value):
    with open(filepath, 'r') as f:
        lines = f.readlines()
    lines[line_idx] = f"{new_value}\n"
    with open(filepath, 'w') as f:
        f.writelines(lines)

def current_date():
    return datetime.date.today().strftime('%y%m%d')

def timestep_sizing(config_path, M, P, f_s, T_sim):
    dt = (1 - M) / (P * f_s)
    total_step = int(np.ceil(T_sim / dt))
    config_val_time = f"{total_step} {dt:.6f}"
    edit_line(config_path, 14, config_val_time)
    return dt

def grid_sizing(config_path, M, X, lambda_s, R):
    dx = (1 - M) * lambda_s / X
    N = int(np.round(2 * np.pi * R / dx))
    config_val_spatial = f"{R} {N} {N}"
    edit_line(config_path, 6, config_val_spatial)
    return N

In [41]:
with open('input_source.txt', 'r') as f:
    lines = f.readlines()[6]
    print(lines)

628.0 1.0 0.0



In [48]:
### Stationary Single Monopole, R/lambda = 1 ### 
config = 'input_config.txt'
source = 'input_source.txt'
default_solver_output = 'FWH_observer_time_01_RT.dat' 
analytical_output = 'FWH_observer_time_01_validation.dat'
date = current_date()


X_targets = [20.0, 30.0, 40.0, 50.0, 60.0, 70.0, 80.0, 90.0, 100.0]
P_targets = [20.0, 30.0, 40.0, 50.0, 60.0, 70.0, 80.0]
M = 0.0
m_str = f"{int(M * 100):02d}" 

c = 340.0
# f_source = 100
# omega = 2 * np.pi * f_source
omega = 100 # rad/s
f_source = omega / (2 * np.pi) # Hz

edit_line(source, 6, f'{omega:.2f} 1.0 0.0')

R = c / f_source # R = lambda
T_sim = 1 / f_source

output_dir = f"data_{date}/single_mono_stationary_noncompact_100rads"
os.makedirs(output_dir, exist_ok=True)

for x in X_targets:
    N = grid_sizing(config, M, x, (c / f_source), R)

    if N * N >= 1e6:
        processes = 8
    elif N * N >= 5e5:
        processes = 4
    elif N * N >= 1e5:
        processes = 2
    else:
        processes = 1
    
    for p in P_targets:
        dt = timestep_sizing(config, M, p, f_source, T_sim)
        print(f"Running Case: P = {p}, X = {x}, M = {M}, Grid Count = {N*N}, Time Step Size = {dt:.6f}, Timestep Count = {np.ceil(T_sim / dt)}")

        subprocess.run(["mpirun", "-np", str(processes), "../bin/FWH_Sphere"], check=True)

        new_filename = f"P{int(p)}_X{int(x)}_M{m_str}.dat"

        destination = os.path.join(output_dir, new_filename)
        shutil.move(default_solver_output, destination)

        if p == P_targets[-1] and x == X_targets[-1]:
            ana_filename = f"ana_M{m_str}.dat"
            ana_destination = os.path.join(output_dir, ana_filename)
            shutil.move(analytical_output, ana_destination)

print("Batch Processing Complete")

Running Case: P = 20.0, X = 20.0, M = 0.0, Grid Count = 15876, Time Step Size = 0.003142, Timestep Count = 21.0
 ... Initializing MPI
 ... MPI initialized successfully
 ... Getting MPI rank and size
 ... Total number of MPI processes: 1
 ... MPI rank of current process: 0
 ... Reading config parameters

 ... reading input parameters from file

 FW-H parameters: 
 Far-field sound speed        :  340.00000 [m/s]
 Far-field air density        :    1.20000 [kg/m^3]
 FW-H surface grid type       : SPHERE               
 FW-H surface grid radius     :   21.36283 [m]
 FW-H surface grid lines      :        126       126 
 FW-H surface grid motion     : SOURCE               
 FW-H surface data type       : SOURCE               
 FW-H integration method      : RETARDED_TIME        
 FW-H kernel type             : TIME_DOMAIN          
 FW-H time integration levels :         21 
 FW-H time stepping           :    0.00314 [s]
 Temporal DFT post-processing : F 
 Debugging level              :      

In [33]:
### New Test, Stationary Monopole ###
date = current_date()
X_targets = [20.0, 30.0, 40.0, 50.0, 60.0, 70.0, 80.0, 90.0, 100.0]
P_targets = [20.0, 30.0, 40.0, 50.0, 60.0, 70.0, 80.0]
M = 0.0
m_str = f"{int(M * 100):02d}" 
R = 1.0
c = 340.0
omega = 10 # rad/s
f_source = omega / (2 * np.pi) # Hz

D_obs = 100.0
# T_sim = ((D_obs / c) + (5 / f_source))
T_sim = 1 / f_source

config = 'input_config.txt'
source = 'input_source.txt'
default_solver_output = 'FWH_observer_time_01_RT.dat' 
analytical_output = 'FWH_observer_time_01_validation.dat'

output_dir = f"data_{date}/single_mono_stationary_R20"
os.makedirs(output_dir, exist_ok=True)

for x in X_targets:
    N = grid_sizing(config, M, x, (c / f_source), R)

    if N * N >= 1e6:
        processes = 8
    elif N * N >= 5e5:
        processes = 4
    elif N * N >= 1e5:
        processes = 2
    else:
        processes = 1
    
    for p in P_targets:
        dt = timestep_sizing(config, M, p, f_source, T_sim)
        print(f"Running Case: P = {p}, X = {x}, M = {M}, Grid Count = {N*N}, Time Step Size = {dt:.6f}, Timestep Count = {np.ceil(T_sim / dt)}")

        subprocess.run(["mpirun", "-np", str(processes), "../bin/FWH_Sphere"], check=True)

        new_filename = f"P{int(p)}_X{int(x)}_M{m_str}.dat"

        destination = os.path.join(output_dir, new_filename)
        shutil.move(default_solver_output, destination)

        if p == P_targets[-1] and x == X_targets[-1]:
            ana_filename = f"ana_M{m_str}.dat"
            ana_destination = os.path.join(output_dir, ana_filename)
            shutil.move(analytical_output, ana_destination)

print("Batch Processing Complete")

Running Case: P = 20.0, X = 20.0, M = 0.0, Grid Count = 1, Time Step Size = 0.031416, Timestep Count = 21.0
 ... Initializing MPI
 ... MPI initialized successfully
 ... Getting MPI rank and size
 ... Total number of MPI processes: 1
 ... MPI rank of current process: 0
 ... Reading config parameters

 ... reading input parameters from file

 FW-H parameters: 
 Far-field sound speed        :  340.00000 [m/s]
 Far-field air density        :    1.20000 [kg/m^3]
 FW-H surface grid type       : SPHERE               
 FW-H surface grid radius     :    1.00000 [m]
 FW-H surface grid lines      :          1         1 
 FW-H surface grid motion     : SOURCE               
 FW-H surface data type       : SOURCE               
 FW-H integration method      : RETARDED_TIME        
 FW-H kernel type             : TIME_DOMAIN          
 FW-H time integration levels :         21 
 FW-H time stepping           :    0.03142 [s]
 Temporal DFT post-processing : F 
 Debugging level              :          

In [ ]:
### P vs X Matrix Gen ###
### NOT DOPPLER SHIFTED ###
M = 0.7
X_targets = [10.0, 20.0, 30.0, 40.0, 50.0, 60.0, 70.0, 80.0, 90.0, 100.0]
P_targets = [10.0, 20.0, 30.0, 40.0, 50.0, 60.0, 70.0, 80.0]

speed_of_sound = 340.0
omega = 10.0
R = 20.0
f_source = omega / (2 * np.pi)
T_sim = 2.5            

config_path = 'input_config.txt'
source_path = 'input_source.txt'
default_solver_output = 'FWH_observer_time_01_RT.dat' 
analytical_output = 'FWH_observer_time_01_validation.dat'
output_dir = 'data_260610/single_mono/'
os.makedirs(output_dir, exist_ok=True)

ux = M * speed_of_sound
source_val = f"{ux}, 0.0, 0.0, 0.0, 0.0, 0.0"
edit_line(source_path, 14, source_val) 
m_str = f"{int(M * 100):02d}" 

for x in X_targets:
    dx = speed_of_sound / (x * f_source)
    N = int(np.round((2 * np.pi * R) / dx))
    
    config_val_spatial = f"{R}, {N}, {N}" 
    edit_line(config_path, 6, config_val_spatial) 
    
    for p in P_targets:
        dt = 1 / (p * f_source)
        total_steps = int(np.ceil(T_sim / dt))
        
        config_val_time = f"{total_steps} {dt:.6f}"
        edit_line(config_path, 14, config_val_time)
        cfl = x / p
        
        print(f"Running Case: M={M:.2f} | CFL={cfl} | N={N} | dt={dt:.6f}")
        
        elements = N * N
        if elements > 1e6:
            processes = 8
        elif elements > 2.5e5:
            processes = 4
        elif elements > 1e5:
            processes = 2
        else:
            processes = 1

        subprocess.run(["mpirun", "-np", str(processes), "../bin/FWH_Sphere"], check=True) 
        
        # cfl_str = f"{int(cfl * 100):03d}"
        new_filename = f"P{int(p)}_X{int(x)}_M{m_str}.dat"
        destination = os.path.join(output_dir, new_filename)
        shutil.move(default_solver_output, destination)

        if p == P_targets[-1] and x == X_targets[0]:
            ana_filename = f"ana_M{m_str}.dat"
            ana_destination = os.path.join(output_dir, ana_filename)
            shutil.copy(analytical_output, ana_destination)

print("Batch processing complete.")


In [ ]:
### Multiple Observer Test Case ###
date = "260610"

M = 0.0
X_targets = [10.0, 20.0, 30.0, 40.0, 50.0, 60.0, 70.0, 80.0, 90.0, 100.0]
P_targets = [10.0, 20.0, 30.0, 40.0, 50.0, 60.0, 70.0, 80.0]

R = 20.0
speed_of_sound = 340.0
omega = 10.0
f_source = omega / (2 * np.pi)

config_path = 'input_config.txt'
source_path = 'input_source.txt'

observers = ['01', '02', '03', '04', '05', '06', '07', '08']
base_output_dir = f'data_{date}/dual_mono/'

output_dir = f'data_{date}/multi_obs/'
os.makedirs(output_dir, exist_ok=True)

for obs in observers:
    os.makedirs(os.path.join(base_output_dir, f"obs_{obs}"), exist_ok=True)

ux = M * speed_of_sound
source_speed = f"{ux} 0.0 0.0 0.0 0.0 0.0"
edit_line(source_path, 14, source_speed)
T_sim = 75.0 / ux if ux > 0.0 else 2.5
m_str = f"{int(M * 100):02d}"

for x in X_targets:
    dx = speed_of_sound * (1 - M) / (x * f_source)
    N = int(np.round(2 * np.pi * R / dx))

    config_val_spatial = f"{R}, {N}, {N}"
    edit_line(config_path, 6, config_val_spatial)

    elements = N * N
    if elements >= 1e6:
        processes = 8
    elif elements >= 5e5:
        processes = 4
    elif elements >= 1e5:
        processes = 2
    else:
        processes = 1

    for p in P_targets:
        dt = (1 - M) / (p * f_source)
        total_steps = int(np.ceil(T_sim / dt))

        config_val_time = f"{total_steps} {dt:.6f}"
        edit_line(config_path, 14, config_val_time)

        cfl = x / p
        print(f"Running Case: M={M:.2f} | X/P={cfl} | N={N} | dt={dt:.6f}")

        subprocess.run(["mpirun", "-np", str(processes), "../bin/FWH_Sphere"], check=True)

        new_filename = f"P{int(p)}_X{int(x)}_M{m_str}.dat"

        for obs in observers:
            solver_file = f"FWH_observer_time_{obs}_RT.dat"
            dest_dir = os.path.join(base_output_dir, f"obs_{obs}")
            destination = os.path.join(dest_dir, new_filename)

            if os.path.exists(solver_file):
                shutil.move(solver_file, destination)

        if p == P_targets[-1] and x==X_targets[-1]:
            for obs in observers:
                ana_file = f"FWH_observer_time_{obs}_validation.dat"
                dest_dir = os.path.join(base_output_dir, f"obs_{obs}")
                ana_destination = os.path.join(dest_dir, f"ana_M{m_str}.dat")

                if os.path.exists(ana_file):
                    shutil.copy(ana_file, ana_destination)

print("Batch Processing Complete")

In [ ]:
### X ANALYSIS ###
mach_numbers = [0.5, 0.6, 0.7, 0.8, 0.9]
cfl_targets = [0.25, 0.5, 0.75, 1.0, 1.25, 1.5]

X_target = 80.0
R = 20.0              
speed_of_sound = 340.0
omega = 10.0
f_source = omega / (2 * np.pi)
T_sim = 2.5            

config_path = 'input_config.txt'
source_path = 'input_source.txt'
default_solver_output = 'FWH_observer_time_01_RT.dat' 
analytical_output = 'FWH_observer_time_01_validation.dat'

output_dir = 'data_260603/test_cfl_PPW80/'
os.makedirs(output_dir, exist_ok=True)

for M in mach_numbers:
    
    ux = M * speed_of_sound
    source_val = f"{ux}, 0.0, 0.0, 0.0, 0.0, 0.0"
    edit_line(source_path, 14, source_val) 
    
    m_str = f"{int(M * 100):02d}" 
    
    dx = speed_of_sound * (1 - M) / (X_target * f_source)
    N = int(np.round((2 * np.pi * R) / dx))
    
    config_val_spatial = f"{R}, {N}, {N}" 
    edit_line(config_path, 6, config_val_spatial) 
    
    for cfl in cfl_targets:
        P_target = X_target / cfl
        dt = (1 - M) / (P_target * f_source)
        total_steps = int(np.ceil(T_sim / dt))
        
        config_val_time = f"{total_steps} {dt:.6f}"
        edit_line(config_path, 14, config_val_time)
        
        print(f"CONSTANT X CASE: \n Running Case: M={M:.2f} | CFL={cfl} | N={N} | dt={dt:.6f}")
        
        if N*N > 1e6:
            processes = 8
        elif N*N > 5e5:
            processes = 4
        elif N*N > 1e5:
            processes = 2
        else:
            processes = 1

        # subprocess.run(["mpirun", "-np", str(processes), "../../bin/FWH_Sphere"], check=True) 
        
        cfl_str = f"{int(cfl * 100):03d}"
        new_filename = f"cfl{cfl_str}_M{m_str}.dat"
        destination = os.path.join(output_dir, new_filename)
        # shutil.move(default_solver_output, destination)

        if cfl == 0.25:
            ana_filename = f"ana_M{m_str}.dat"
            ana_destination = os.path.join(output_dir, ana_filename)
            # shutil.move(analytical_output, ana_destination)

print("Batch processing complete.")

In [ ]:
### P ANALYSIS ###
mach_numbers = [0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95]
cfl_targets = [0.25, 0.5, 0.75, 1.0, 1.25, 1.5]

P_target = 15.0
R = 20.0
speed_of_sound = 340.0
omega = 10.0
f_source = omega / (2 * np.pi)
T_sim = 2.5

config_path = 'input_config.txt'
source_path = 'input_source.txt'
default_solver_output = 'FWH_observer_time_01_RT.dat' 
analytical_output = 'FWH_observer_time_01_validation.dat'
output_dir = 'data_260528/test_cfl_PPP10/'

os.makedirs(output_dir, exist_ok=True)

for M in mach_numbers:
    ux = M * speed_of_sound
    source_val = f"{ux}, 0.0, 0.0, 0.0, 0.0, 0.0"
    edit_line(source_path, 14, source_val)

    m_str = f"{int(M * 100):02d}"
    dt = (1 - M) / (P_target * f_source)
    N_step = int(np.ceil(T_sim / dt))

    config_val_time = f"{N_step}, {dt:.6f}"
    edit_line(config_path, 14, config_val_time)

    for cfl in cfl_targets:
        X_target = cfl * P_target
        dx = speed_of_sound * (1 - M) / (X_target * f_source)
        N = int(np.round(2 * np.pi * R / dx))
        config_val_spatial = f"{R}, {N}, {N}"
        edit_line(config_path, 6, config_val_spatial)

        print(f"Running Case= M={M:.2f} | CFL={cfl} | N={N} | dt={dt:.6f}")

        if M == 0.95:
            processes = 2
        # elif M >= 0.75:
            # processes = 2
        else:
            processes = 1

        subprocess.run(["mpirun", "-np", str(processes), "../bin/FWH_Sphere"], check=True)

        cfl_str = f"{int(cfl * 100):03d}"
        new_filename = f"cfl{cfl_str}_M{m_str}.dat"
        destination = os.path.join(output_dir, new_filename)
        shutil.move(default_solver_output, destination)

        if cfl == 0.25:
            ana_filename = f"ana_M{m_str}.dat"
            ana_destination = os.path.join(output_dir, ana_filename)
            shutil.move(analytical_output, ana_destination)

print("Batch processing complete.")

In [ ]:

# mach_numbers = [0.95]
# R_targets = [150.0]

# X_target = 20.0        
# CFL_target = 1.0       
# P_target = X_target / CFL_target

# speed_of_sound = 340.0
# omega = 10.0
# f_source = omega / (2 * np.pi)
# T_sim = 2.5            

# config_path = 'input_config.txt'
# source_path = 'input_source.txt'
# default_solver_output = 'FWH_observer_time_01_RT.dat' 
# analytical_output = 'FWH_observer_time_01_validation.dat'
# output_dir = 'data_260601/test_radius/'

# os.makedirs(output_dir, exist_ok=True)
# for M in mach_numbers:
#     ux = M * speed_of_sound
#     source_val = f"{ux}, 0.0, 0.0, 0.0, 0.0, 0.0"
#     edit_line(source_path, 14, source_val) 
    
#     m_str = f"{int(M * 100):02d}" 
    
#     dt = (1 - M) / (P_target * f_source)
#     N_step = int(np.ceil(T_sim / dt))
    
#     config_val_time = f"{N_step} {dt:.6f}"
#     edit_line(config_path, 14, config_val_time)
    
#     dx = speed_of_sound * (1 - M) / (X_target * f_source)
    
#     for R in R_targets:
#         N = int(np.round((2 * np.pi * R) / dx))
        
#         config_val_spatial = f"{R}, {N}, {N}" 
#         edit_line(config_path, 6, config_val_spatial) 
        
#         print(f"Running Case: M={M:.2f} | R={R} | N={N} | dt={dt:.6f}")
        
#         elements = N * N
#         if elements > 1e6:
#             processes: 8
#         # elif elements > 5e5: 
#             # processes = 8
#         elif elements > 2.5e5:
#             processes = 4
#         elif elements > 1e5:
#             processes = 2
#         else:
#             processes = 1

#         # subprocess.run(["mpirun", "-np", str(processes), "../../bin/FWH_Sphere"], check=True) 
        
#     #     r_str = f"{int(R):03d}"
#     #     new_filename = f"r{r_str}_M{m_str}.dat"
#     #     destination = os.path.join(output_dir, new_filename)
#     #     shutil.move(default_solver_output, destination)
        
#     # ana_filename = f"ana_M{m_str}.dat"
#     # ana_destination = os.path.join(output_dir, ana_filename)
#     # shutil.move(analytical_output, ana_destination)

# print("Batch processing complete.")

In [7]:
20 * (np.log(1e-8 / 2e-5) - np.log(1e-16 / 2e-5))

np.float64(368.4136148790474)

In [ ]:
# M = 0.7 #[0.5, 0.7, 0.9]
# X_targets = [10.0, 20.0, 30.0, 40.0, 50.0, 60.0, 70.0, 80.0, 90.0, 100.0]
# P_targets = [10.0, 20.0, 30.0, 40.0, 50.0, 60.0, 70.0, 80.0]

# speed_of_sound = 340.0
# omega = 10.0
# R = 20.0
# f_source = omega / (2 * np.pi)
# T_sim = 2.5            

# config_path = 'input_config.txt'
# source_path = 'input_source.txt'
# default_solver_output = 'FWH_observer_time_01_RT.dat' 
# analytical_output = 'FWH_observer_time_01_validation.dat'
# output_dir = 'data_260603/test_M07/'

# os.makedirs(output_dir, exist_ok=True)

# ux = M * speed_of_sound
# source_val = f"{ux}, 0.0, 0.0, 0.0, 0.0, 0.0"
# edit_line(source_path, 14, source_val) 
# m_str = f"{int(M * 100):02d}" 


# for x in X_targets:
#     dx = speed_of_sound * (1 - M) / (x * f_source)
#     N = int(np.round((2 * np.pi * R) / dx))
    
#     config_val_spatial = f"{R}, {N}, {N}" 
#     edit_line(config_path, 6, config_val_spatial) 
    
#     for p in P_targets:
#         dt = (1 - M) / (p * f_source)
#         total_steps = int(np.ceil(T_sim / dt))
        
#         config_val_time = f"{total_steps} {dt:.6f}"
#         edit_line(config_path, 14, config_val_time)
#         cfl = x / p
        
#         print(f"Running Case: M={M:.2f} | CFL={cfl} | N={N} | dt={dt:.6f}")
        
#         elements = N * N
#         if elements > 1e6:
#             processes = 8
#         elif elements > 2.5e5:
#             processes = 4
#         elif elements > 1e5:
#             processes = 2
#         else:
#             processes = 1

#         subprocess.run(["mpirun", "-np", str(processes), "../bin/FWH_Sphere"], check=True) 
        
#         # cfl_str = f"{int(cfl * 100):03d}"
#         new_filename = f"P{int(p)}_X{int(x)}_M{m_str}.dat"
#         destination = os.path.join(output_dir, new_filename)
#         shutil.move(default_solver_output, destination)

#         if p == P_targets[-1] and x == X_targets[0]:
#             ana_filename = f"ana_M{m_str}.dat"
#             ana_destination = os.path.join(output_dir, ana_filename)
#             shutil.copy(analytical_output, ana_destination)

# print("Batch processing complete.")

In [ ]:
# M = 0.9 #[0.5, 0.7, 0.9]
# X_targets = [10.0, 20.0, 30.0, 40.0, 50.0, 60.0, 70.0, 80.0, 90.0, 100.0]
# P_targets = [10.0, 20.0, 30.0, 40.0, 50.0, 60.0, 70.0, 80.0]

# speed_of_sound = 340.0
# omega = 10.0
# R = 20.0
# f_source = omega / (2 * np.pi)
# T_sim = 2.5            

# config_path = 'input_config.txt'
# source_path = 'input_source.txt'
# default_solver_output = 'FWH_observer_time_01_RT.dat' 
# analytical_output = 'FWH_observer_time_01_validation.dat'
# output_dir = 'data_260603/test_M09/'

# os.makedirs(output_dir, exist_ok=True)

# ux = M * speed_of_sound
# source_val = f"{ux}, 0.0, 0.0, 0.0, 0.0, 0.0"
# edit_line(source_path, 14, source_val) 
# m_str = f"{int(M * 100):02d}" 


# for x in X_targets:
#     dx = speed_of_sound * (1 - M) / (x * f_source)
#     N = int(np.round((2 * np.pi * R) / dx))
    
#     config_val_spatial = f"{R}, {N}, {N}" 
#     edit_line(config_path, 6, config_val_spatial) 
    
#     for p in P_targets:
#         dt = (1 - M) / (p * f_source)
#         total_steps = int(np.ceil(T_sim / dt))
        
#         config_val_time = f"{total_steps} {dt:.6f}"
#         edit_line(config_path, 14, config_val_time)
#         cfl = x / p
        
#         print(f"Running Case: M={M:.2f} | CFL={cfl} | N={N} | dt={dt:.6f}")
        
#         elements = N * N
#         if elements > 1e6:
#             processes = 8
#         elif elements > 2.5e5:
#             processes = 4
#         elif elements > 1e5:
#             processes = 2
#         else:
#             processes = 1

#         subprocess.run(["mpirun", "-np", str(processes), "../bin/FWH_Sphere"], check=True) 
        
#         # cfl_str = f"{int(cfl * 100):03d}"
#         new_filename = f"P{int(p)}_X{int(x)}_M{m_str}.dat"
#         destination = os.path.join(output_dir, new_filename)
#         shutil.move(default_solver_output, destination)

#         if p == P_targets[-1] and x == X_targets[0]:
#             ana_filename = f"ana_M{m_str}.dat"
#             ana_destination = os.path.join(output_dir, ana_filename)
#             shutil.copy(analytical_output, ana_destination)

# print("Batch processing complete.")

In [ ]:
# import os
# import shutil
# import subprocess

# mach_numbers = [0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95]
# radii = [50, 100, 200]
# grids = {
#     'c': '20, 20', 
#     'm': '40, 40', 
#     'f': '60, 60'  
# }

# time = {
#     'c': '125 0.02',
#     'm': '250 0.01',
#     'f': '500 0.005'
# }

# speed_of_sound = 340.0 # m/s (170 / 0.5)

# config_path = 'input_config.txt'
# source_path = 'input_source.txt'
# default_solver_output = 'FWH_observer_time_01_RT.dat' 
# output_dir = 'Data 260527/test_mach/'

# os.makedirs(output_dir, exist_ok=True)

# # 2. File Editing Helper
# def edit_line(filepath, line_idx, new_value):
#     """Directly overwrites a specific line index."""
#     with open(filepath, 'r') as f:
#         lines = f.readlines()
    
#     lines[line_idx] = f"{new_value}\n"
    
#     with open(filepath, 'w') as f:
#         f.writelines(lines)


# for M in mach_numbers:

#     ux = M * speed_of_sound
#     source_val = f"{ux}, 0.0, 0.0, 0.0, 0.0, 0.0"
#     edit_line(source_path, 14, source_val) 
    
#     m_str = str(M).replace('.', '')
    
#     # for r in radii:
#     #     for grid_label, grid_res in grids.items():
            
#     #         # Inject radius and grid into config 
#     #         config_val = f"{r}, {grid_res}"
#     #         edit_line(config_path, 6, config_val) 
            
#     #         print(f"Running Case: R={r} | M={M} | Grid={grid_label.upper()}")
            
#     #         # Execute the compiled Fortran binary
#     #         subprocess.run(["../../bin/FWH_Sphere"], check=True) 
            
#     #         print(f"Renaming and moving file r{r}_M{m_str}{grid_label}.dat")
#     #         new_filename = f"r{r}_M{m_str}{grid_label}.dat"
#     #         destination = os.path.join(output_dir, new_filename)
            
#     #         shutil.move(default_solver_output, destination)

#     for time_label, time_res in time.items():
#         config_val = f"{time_res}"
#         edit_line(config_path, 14, config_val)

#         print(f'Running Case: timestep size={time_res} | M={M}')
#         # subprocess.run(["../../bin/FWH_Sphere"], check=True) 

# #         # Generate the new filename: r200_M09f.dat
#         print(f"Renaming and moving file t_M{m_str}{time_label}.dat")
#         new_filename = f"t_M{m_str}{time_label}.dat"
#         destination = os.path.join(output_dir, new_filename)
#         # shutil.move(default_solver_output, destination)

# print("Batch processing complete.")